# 26 — MCP Resources and Prompts

Parts 1-2 of this course were almost entirely about **tools** -- the Azure
MCP Server exposes dozens of namespaces, and every one of them is a tool.
MCP defines two more primitives the Azure MCP Server barely touches:
**resources** (readable, URI-addressed data) and **prompts** (reusable
message templates). This notebook exercises both against
`examples/mcp_foundations/custom_server.py`.

## Resources: data, not actions

Resources are **application-driven** -- the host decides how to surface
them (a picker UI, automatic inclusion, search), not the model choosing to
"call" one the way it calls a tool. Servers that support resources declare
the capability, optionally with `subscribe` (per-item change
notifications) and `listChanged` (list-change notifications):

```json
{ "capabilities": { "resources": { "subscribe": true, "listChanged": true } } }
```

Protocol messages: `resources/list` (paginated), `resources/read`, and --
for parameterized resources like our server's `greeting://{name}` --
`resources/templates/list`, which returns
[URI templates](https://datatracker.ietf.org/doc/html/rfc6570) instead of
concrete URIs.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from pydantic import AnyUrl
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_script = str(Path.cwd().parent / "examples" / "mcp_foundations" / "custom_server.py")
server_params = StdioServerParameters(command=sys.executable, args=[server_script])

async def read_resources():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            fixed = await session.list_resources()
            templated = await session.list_resource_templates()
            content = await session.read_resource(AnyUrl("greeting://World"))
            return fixed, templated, content

fixed, templated, content = await read_resources()
print("fixed resources:", [str(r.uri) for r in fixed.resources])
print("resource templates:", [t.uriTemplate for t in templated.resourceTemplates])
print("greeting://World contents:", content.contents[0].text)

## Prompts: user-controlled templates

Prompts are **user-controlled** -- typically surfaced as something the
user explicitly invokes, like a slash command, rather than something the
model decides to use on its own. That's a three-way split worth
remembering:

| Primitive | Who decides to use it | Analogy |
| --- | --- | --- |
| Tool | The model (function calling) | A POST endpoint |
| Resource | The host application | A GET endpoint |
| Prompt | The user, explicitly | A slash command |

A prompt definition carries a `name`, optional `description`, and a list of
`arguments`; `prompts/get` returns one or more `PromptMessage` objects whose
content can be text, image, audio, or an embedded resource.

In [ ]:
async def get_prompt():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            prompts = await session.list_prompts()
            result = await session.get_prompt(
                "greet_user", arguments={"name": "Ada", "style": "formal"}
            )
            return prompts, result

prompts, result = await get_prompt()
print("available prompts:", [(p.name, [a.name for a in (p.arguments or [])]) for p in prompts.prompts])
print("rendered prompt message:", result.messages[0].content.text)

## When to reach for which primitive

- Building something the **model should decide to invoke** based on the
  conversation? → **tool** (everything in Parts 1-2).
- Exposing **data the host should decide to include** as context, with or
  without the model asking? → **resource**.
- Offering a **reusable instruction the user explicitly triggers**? →
  **prompt**.

This mirrors the "which policy do I actually need" framing from notebook
23 -- name the specific primitive your scenario calls for, rather than
reaching for whichever one you already know.

Continue to
[27_mcp_sampling_and_roots.ipynb](27_mcp_sampling_and_roots.ipynb) for the
two primitives that flip the direction of the conversation: the *server*
asking the *client* for help.